In [ ]:
from pycromanager import Core, Studio
import time
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
import bh_spc
from bh_spc import spcm
import pprint
from collections import OrderedDict
import datetime

def print_time():
    print(datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

def obj_2_list(name):
    """Convert Java object to Python list."""
    return [name.get(i) for i in range(name.size())]

def print_params(params):
    params_ = OrderedDict(params)
    for k in params_:
        print(f'{k:<20} : {params_[k]:.3f}')



def get_microtimes_MT(duration = 0.1, buf_size = 32768, mod_no=0):

    spcm.start_measurement(mod_no)
    start_time = time.monotonic()

    data = []  # Collect arrays of data into a list.
    while True:
        elapsed = time.monotonic() - start_time
        if elapsed >= duration:
            spcm.stop_measurement(mod_no)
            break
        buf = spcm.read_fifo_to_array(mod_no, buf_size)
        if len(buf):
            data.append(buf)
        if len(buf) < buf_size:  # We've read all there is to read.
            time.sleep(0.001)

    # Make sure to read the data that arrived after stopping (if you need it).
    while True:
        buf = spcm.read_fifo_to_array(mod_no, buf_size)
        if not len(buf):
            break
        data.append(buf)

    records = np.concatenate(data).view(np.uint32)
    len(records)

    had_gap = np.any(np.bitwise_and(records, 1 << 29))
    print("There was {} gap".format("a" if had_gap else "no"))

    photons = np.extract(np.bitwise_and(records, 0b1001 << 28) == 0, records)
    len(photons)

    max_12bit = (1 << 12) - 1  # 4095
    microtimes = np.bitwise_and(np.right_shift(photons, 16), max_12bit)

    # Reverse the microtimes by subtracting from the max value, because the raw
    # microtime is measured from photon to SYNC, not SYNC to photon.
    microtimes = max_12bit - microtimes
    
    return microtimes,photons


core = Core()
studio = Studio()

core.set_roi(0,0,20,20)
r = core.get_roi()
print(r.width, r.height, r.x, r.y)

core.set_property("OSc-LSM","Dev1-Fermat Spiral Scan","Yes")
core.get_property("OSc-LSM","Dev1-Fermat Spiral Scan")


scan_duration = 5 # sec  ## controlled by time.sleep

if core.is_sequence_running(): core.stop_sequence_acquisition()

studio.live().set_live_mode_on(True)
for i in range(scan_duration):
    print(f"[{i+1}/4]:", studio.live().is_live_mode_on(), core.is_sequence_running())
    time.sleep(1)
studio.live().set_live_mode_on(False)


spcm.init(r'C:\Program Files\Micro-Manager-2.0\spcm.ini')

mod_no = 0 # SPC 180NX
spcm.get_init_status(mod_no)

params = spcm.get_parameters(mod_no)

params.collect_time = scan_duration # sec
spcm.set_parameters(0, params)

duration = spcm.get_parameter(mod_no,spcm.ParID.COLLECT_TIME)  # s
buf_size = 2**15  # Max number of 16-bit words in a single read.

print(duration, buf_size)


for k in range(3):
    print_time()
    mt,ph = get_microtimes_MT(duration,buf_size)
    print(len(mt),len(ph))
    x,h = np.histogram(mt,64)
    plt.plot(h[:-1],x)
    print(np.argmax(x))

duration

duration_s = 10

